# 05 · De LeNet a AlexNet
*Apilar convolución, ReLU y pooling, entrenar en CIFAR-10, y comparar con el MLP*

Notebook del episodio 5 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/de-lenet-a-alexnet)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** LeNet-5 (paper: 340.908 conexiones, 60.000 parámetros, 0,95 % de error en dígitos) → formas 6×28×28 … 16×5×5 → 400 → 120 → 84 → 10, 62.006 parámetros (fc1 = 78 %) → loop de entrenamiento corto (1 epoch, 10.000 imágenes): pérdida 2,30 → 1,67, test 40,5 % → preparar_ep05.py (Adam 1e-3, 12 epochs): MLP 53,8 %, LeNet 63,5 %, CNNChica 74,7 % con 100.874 parámetros (train 88,9 %) → primer gato de test: 94 % gato, L = 0,06 → píxeles mezclados: MLP 52,9 %, CNN 50,1 % (sesgo inductivo) → confusión: 268/1000 perros → gato → AlexNet: 60 M parámetros, ReLU 6× más rápido en CIFAR-10, dropout, 2 GTX 580, top-5 ILSVRC-2012 15,3 % (7 redes) vs. 26,2 %

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio cinco: de LeNet a AlexNet. Ya tenemos todas las piezas: convolución, canales, pooling. Hoy las apilamos en una red completa, la entrenamos en CIFAR diez, el dataset del Canadian Institute for Advanced Research, y la comparamos con el MLP, multilayer perceptron, del episodio uno, que llegaba a cincuenta y cuatro por ciento.*

### Qué vamos a ver

- La receta: conv → ReLU → pool, varias veces, y densas al final
- LeNet-5 (1998), línea por línea
- Entrenar en CIFAR-10 y comparar con el MLP
- AlexNet (2012): la misma receta, a otra escala

El plan. La receta de una red convolucional, CNN, por convolutional neural network: convolución, activación ReLU, rectified linear unit, y pooling, repetido, y unas capas densas al final que deciden la clase. El primer ejemplo famoso, LeNet cinco, de mil novecientos noventa y ocho. Entrenamos dos redes así en CIFAR diez y las comparamos con el MLP. Y AlexNet, que en dos mil doce aplicó la misma receta a una escala nunca vista.

## Parte 1 · LeNet-5

La red con la que Yann LeCun y sus coautores leían dígitos escritos a mano, por ejemplo en cheques.

> **LeCun, Bottou, Bengio & Haffner, 1998 — Gradient-Based Learning Applied to Document Recognition** · [fuente](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf)
>
> **340.908 conexiones · 60.000 parámetros** · «The network in figure 2 contains 340,908 connections, but only 60,000 trainable free parameters because of the weight sharing.» Error en test de LeNet-5: 0,95 %.

El paper es de LeCun, Bottou, Bengio y Haffner. La frase clave es sobre los pesos compartidos: la red tiene trescientas cuarenta mil novecientas ocho conexiones, pero solo sesenta mil parámetros entrenables, porque cada kernel se reusa en toda la imagen. Es lo que medimos en el episodio tres. En su figura de resultados, LeNet cinco se equivoca en el cero coma noventa y cinco por ciento del test de dígitos.

### LeNet-5, de la imagen a la clase

La arquitectura, en el orden en que pasa la imagen. Entra una imagen de treinta y dos por treinta y dos; el original tenía un canal, la nuestra tres. Seis kernels de cinco por cinco: seis mapas de veintiocho por veintiocho. Pooling: catorce por catorce. Dieciséis kernels de cinco: dieciséis mapas de diez por diez. Pooling otra vez: cinco por cinco. Se aplana y pasa por dos capas densas, de ciento veinte y ochenta y cuatro. Y diez números de salida. El original usaba tanh y un promedio con peso; nosotros, ReLU y max pooling, como se hace hoy.

imagen 3×32×32 → conv1 6×28×28 → pool 6×14×14 → conv2 16×10×10 → pool 16×5×5 → densas 120 · 84 → 10 logits

### LeNet en PyTorch, capa por capa

La clase LeNet está en el módulo de redes del lab; acá le seguimos la forma a una imagen. Creamos la red. Recorremos sus dos convoluciones. Cada una va con ReLU, el máximo entre cero y el valor. Y después, max pooling de dos. Al final, aplanamos y contamos los parámetros. Las formas coinciden con el diagrama: seis por veintiocho, seis por catorce, dieciséis por diez, dieciséis por cinco, cuatrocientos números aplanados. Y en total, sesenta y dos mil seis parámetros, cerca de los sesenta mil del paper: el original tenía un solo canal de entrada, y su segunda convolución no conectaba cada mapa con todos los anteriores.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_lenet.py
__file__ = 'snippets/ep05_lenet.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn.functional as F
from _redes import LeNet, contar_parametros

torch.manual_seed(0)

In [3]:
red = LeNet()                               # LeNet-5 para 3 canales
x = torch.randn(1, 3, 32, 32)
for nombre, capa in [("conv1", red.conv1), ("conv2", red.conv2)]:
    x = F.relu(capa(x))
    print(f"{nombre} → {tuple(x.shape[1:])}")
    x = F.max_pool2d(x, 2)
    print(f"pool  → {tuple(x.shape[1:])}")
print(f"flat  → {x[0].numel()} → 120 → 84 → 10 · {contar_parametros(red):,}")

conv1 → (6, 28, 28)
pool  → (6, 14, 14)
conv2 → (16, 10, 10)
pool  → (16, 5, 5)
flat  → 400 → 120 → 84 → 10 · 62,006


### Dónde están los parámetros de LeNet

$$
32 - 5 + 1 = 28 \;\to\; 14 \;\to\; 14 - 5 + 1 = 10 \;\to\; 5
$$
<small>la fórmula de tamaño, con k = 5 y sin padding</small>

$$
C_{out}(C_{in}k^2 + 1):\ \ 6(3\cdot 25+1) = 456,\ \ 16(6\cdot 25+1) = 2416
$$
<small>las dos convoluciones</small>

$$
400 \cdot 120 + 120 = 48\,120
$$
<small>la primera densa, sola, es el 78 %</small>

De dónde salen los números. Los tamaños son la fórmula del episodio dos: treinta y dos menos cinco más uno da veintiocho, el pooling divide por dos, y así. Los parámetros de cada convolución son C out por C in por k al cuadrado más uno: cuatrocientos cincuenta y seis y dos mil cuatrocientos dieciséis. En el dibujo, la suma de todas las capas: las convoluciones casi no se ven. La primera densa, de cuatrocientos a ciento veinte, tiene cuarenta y ocho mil ciento veinte: ella sola es el setenta y ocho por ciento de la red. Las convoluciones hacen casi todo el trabajo con casi nada de pesos.

## Parte 2 · Entrenar en CIFAR-10

Ahora una red un poco más ancha, con tres convoluciones de treinta y dos, sesenta y cuatro y sesenta y cuatro canales, y una sola densa al final. Cien mil parámetros.

### El loop de entrenamiento

Una versión corta: un solo epoch sobre diez mil imágenes. Theta son todos los pesos. Para cada batch de ciento veintiocho imágenes, la pérdida es la entropía cruzada: menos el logaritmo de la probabilidad que la red le da a la clase correcta. Backward calcula, con backpropagation, el gradiente: cuánto sube la pérdida si movemos cada peso. El optimizador Adam, adaptive moment estimation, mueve los pesos en contra del gradiente, escalado por eta, el learning rate, de cero coma cero cero uno. Y medimos la precisión: los corchetes valen uno si la predicción, y con sombrero, acierta, y N es la cantidad de imágenes de test. La pérdida baja de dos coma treinta, que es adivinar entre diez, a uno coma sesenta y siete, y ya acierta el cuarenta por ciento. La versión completa corre doce epochs sobre las cincuenta mil.

**Cada línea, su término:**

- `F.cross_entropy(red(xb), yb)` → $L = -\log p_\theta(y \mid x)$ — entropía cruzada
- `perdida.backward()` → $g = \nabla_\theta L$ — backpropagation: el gradiente de cada peso
- `opt.step()` → $\theta \leftarrow \theta - \eta\, \hat{g}$ — Adam: $\hat g$ es el gradiente reescalado; $\eta$ = learning rate
- `evaluar(red, Xt, yt)` → $\text{precisión} = \tfrac{1}{N}\textstyle\sum_n [\hat y_n = y_n]$ — sobre imágenes que no vio

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_entrenar.py
__file__ = 'snippets/ep05_entrenar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn.functional as F
from _cifar import cargar, normalizar
from _redes import CNNChica, evaluar

torch.manual_seed(0)
X, y = cargar("train", n=10_000)
Xt, yt = cargar("test", n=2_000)
X, Xt = normalizar(X), normalizar(Xt)

In [5]:
red = CNNChica()
opt = torch.optim.Adam(red.parameters(), lr=1e-3)
for paso, i in enumerate(range(0, len(X), 128)):    # 1 epoch de 10.000
    xb, yb = X[i:i + 128], y[i:i + 128]
    perdida = F.cross_entropy(red(xb), yb)       # logits vs. etiqueta
    opt.zero_grad()
    perdida.backward()                           # gradientes
    opt.step()                                   # actualizar pesos
    if paso in (0, 40, 78):
        print(f"paso {paso:2d}: pérdida {perdida.item():.2f}")
print(f"test tras 1 epoch chica: {evaluar(red, Xt, yt):.1%}")

paso  0: pérdida 2.30


paso 40: pérdida 2.01


paso 78: pérdida 1.67


test tras 1 epoch chica: 40.5%


### La pérdida de una imagen

$$
p = \mathrm{softmax}(z),\quad p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}
$$
<small>z: los 10 logits</small>

$$
L = -\log p_{\text{gato}} = -\log 0{,}942 = 0{,}06
$$
<small>cerca de 0: confía en la clase correcta</small>

Una imagen concreta: el mismo gato del episodio uno. La red da diez logits, z, y softmax los convierte en probabilidades: e a la z sub k, dividido la suma de todas. En el dibujo, la red ya entrenada le da noventa y cuatro por ciento a gato. El MLP le daba cuarenta. La pérdida es menos el logaritmo de esa probabilidad: cero coma cero seis. En el dibujo, la curva del logaritmo: cerca de uno es casi plana y vale casi cero, y cerca de cero se hunde. Con el cuarenta por ciento del MLP, la pérdida sería cero coma noventa y dos. Por eso equivocarse con confianza cuesta tanto.

### Precisión en test, epoch por epoch

Las tres redes, entrenadas con el mismo optimizador. El MLP se estanca en cincuenta y cuatro. LeNet, con menos de la vigésima parte de los parámetros, pasa el sesenta y tres. Y la CNN chica, con tres convoluciones, ronda el setenta y cinco desde el epoch nueve, y ahí se aplana: de acá en más, mejora en train pero no en test.

### Las tres redes, medidas igual

Las medimos con el mismo código. Las tres redes guardadas. Cargamos cada una. Contamos sus parámetros. Y comparamos test con train. El MLP: un millón setecientos mil parámetros, cincuenta y tres coma ocho. LeNet: sesenta y dos mil, sesenta y tres coma cinco. La CNN chica: cien mil ochocientos setenta y cuatro, setenta y cuatro coma siete. Diecisiete veces menos parámetros que el MLP y veintiún puntos más. Ojo con train: ochenta y ocho coma nueve. La brecha con test es overfitting, y hay técnicas para achicarla más adelante en la serie.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_comparar.py
__file__ = 'snippets/ep05_comparar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar, normalizar, ruta_lab
from _redes import cargar_modelo, contar_parametros, evaluar

Xt, yt = cargar("test")

In [7]:
Xt = normalizar(Xt)
for archivo in ("mlp_ep01", "lenet_ep05", "cnn_ep05"):
    red, info = cargar_modelo(ruta_lab("datos", archivo + ".pt"))
    print(f"{archivo:10s} {contar_parametros(red):>9,} parámetros"
          f"  test {evaluar(red, Xt, yt):.1%}"
          f"  train {info['precision_train']:.1%}")

mlp_ep01   1,707,274 parámetros  test 53.8%  train 76.2%


lenet_ep05    62,006 parámetros  test 63.5%  train 71.3%


cnn_ep05     100,874 parámetros  test 74.7%  train 88.9%


In [8]:
# Verificación extra (corrió junto con el video)
hist = {a: cargar_modelo(ruta_lab("datos", a + ".pt"))[1]["historia"] for a in ("mlp_ep01", "lenet_ep05", "cnn_ep05")}
json.dump(hist, open(ruta_lab("datos", "figs", "ep05_historia.json"), "w"))

## Parte 3 · Lo que la convolución sabe de antemano

¿De dónde sale la ventaja? Repitamos el experimento de mezclar los píxeles.

### Mezclar los píxeles, ahora con una CNN

Las mismas dos versiones de cada red: una entrenada con las imágenes normales, otra entrenada desde cero con los píxeles mezclados. La misma mezcla del episodio uno. Aplicada al test. Y lo hacemos para el MLP y para la CNN chica. Al MLP le da casi igual: cincuenta y tres coma ocho contra cincuenta y dos coma nueve. La CNN cae de setenta y cuatro coma siete a cincuenta coma uno: peor que el MLP. Mezclada, la imagen ya no tiene bordes ni vecinos, y lo que la convolución supone deja de ser cierto.

In [9]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_permutar.py
__file__ = 'snippets/ep05_permutar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar, normalizar, permutacion, permutar, ruta_lab
from _redes import cargar_modelo, evaluar

Xt, yt = cargar("test")

In [10]:
perm = permutacion(semilla=0)          # la misma mezcla del episodio 1
normal, mezclado = normalizar(Xt), normalizar(permutar(Xt, perm))
for nombre, a, b in [("MLP", "mlp_ep01", "mlp_perm_ep01"),
                     ("CNN", "cnn_ep05", "cnn_perm_ep05")]:
    r_a, _ = cargar_modelo(ruta_lab("datos", a + ".pt"))
    r_b, _ = cargar_modelo(ruta_lab("datos", b + ".pt"))
    print(f"{nombre}: píxeles en orden {evaluar(r_a, normal, yt):.1%}"
          f" · mezclados {evaluar(r_b, mezclado, yt):.1%}")

MLP: píxeles en orden 53.8% · mezclados 52.9%


CNN: píxeles en orden 74.7% · mezclados 50.1%


### Sesgo inductivo: lo que la red supone

- Localidad: lo importante está en píxeles vecinos
- Pesos compartidos: un borde es un borde en cualquier lugar
- Con imágenes reales, el supuesto ayuda: +21 puntos
- Con píxeles mezclados, estorba: −25 puntos

Eso tiene nombre: sesgo inductivo, lo que la arquitectura supone del problema antes de ver un solo dato. La convolución supone localidad: que lo que importa se arma con píxeles vecinos. Y supone que el mismo detector sirve en toda la imagen. En fotos reales esos supuestos son ciertos, y valen más que un millón y medio de parámetros extra. Cuando los rompemos, la CNN pierde veinticinco puntos. Ninguna red es mejor en todo: es mejor en los datos que se parecen a lo que supone.

## Parte 4 · Dónde se equivoca

Un porcentaje solo no dice qué confunde la red. Para eso está la matriz de confusión.

### La matriz de confusión

Una matriz de diez por diez. Predecimos las diez mil imágenes de test. Para cada imagen sumamos uno en la fila de su clase verdadera y la columna de lo que dijo la red. La diagonal son los aciertos; dividida por el total de cada fila, el acierto por clase. Y miramos gato contra perro. Las clases más fáciles son barco y auto, ochenta y ocho y ochenta y cinco por ciento. Las más difíciles, perro y pájaro. Y lo más llamativo: doscientos sesenta y ocho de mil perros los dice gato. Al revés, solo noventa y dos.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_confusion.py
__file__ = 'snippets/ep05_confusion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
from _cifar import CLASES, cargar, normalizar, ruta_lab
from _redes import cargar_modelo, predecir

Xt, yt = cargar("test")

In [12]:
red, _ = cargar_modelo(ruta_lab("datos", "cnn_ep05.pt"))
pred = predecir(red, normalizar(Xt))
M = torch.zeros(10, 10, dtype=torch.long)    # fila: verdad; col: predicción
M.index_put_((yt, pred), torch.ones_like(yt), accumulate=True)
acierto = M.diag() / M.sum(1)
for c in acierto.argsort(descending=True)[[0, 1, -2, -1]].tolist():
    print(f"{CLASES[c]:8s} {acierto[c]:.0%}")
gato, perro = CLASES.index("gato"), CLASES.index("perro")
print(f"gatos que dijo perro: {M[gato, perro]} de 1000")
print(f"perros que dijo gato: {M[perro, gato]} de 1000")

barco    88%
auto     85%
perro    60%
pájaro   54%
gatos que dijo perro: 92 de 1000
perros que dijo gato: 268 de 1000


In [13]:
# Verificación extra (corrió junto con el video)
with torch.no_grad():
    z = red(normalizar(Xt[:1]))[0]
json.dump({"M": M.tolist(), "acierto": [round(float(a), 3) for a in acierto], "logits0": [round(float(v), 2) for v in z],
           "probs0": [round(float(v), 3) for v in z.softmax(0)], "clase0": CLASES[int(yt[0])]}, open(ruta_lab("datos", "figs", "ep05_confusion.json"), "w"))

### Verdad (filas) contra predicción (columnas)

La matriz entera. La diagonal brilla: cada fila tiene mil imágenes y la mayoría cae en su lugar. Fuera de la diagonal, mirá la columna de gato: recibe errores de perro, pero también de pájaro, ciervo y rana. Una lectura posible: a treinta y dos por treinta y dos, para esta red, un animal peludo y borroso tiende a ser gato. Y los vehículos se confunden entre ellos: auto con camión, avión con barco.

## Parte 5 · AlexNet, 2012

Catorce años después de LeNet, la misma receta ganó la competencia más grande de clasificación de imágenes.

### Qué cambió en AlexNet

- 5 convoluciones + 3 densas: 60 millones de parámetros
- ReLU en vez de tanh: entrena varias veces más rápido
- Dropout contra el overfitting de las densas
- Dos GPU GTX 580: entre cinco y seis días

Las piezas son las mismas; cambió la escala y algunos trucos. Cinco capas convolucionales y tres densas, con sesenta millones de parámetros, para imágenes de alta resolución de mil clases. ReLU en lugar de tanh: según el paper, una red chica en CIFAR diez llegó a veinticinco por ciento de error de entrenamiento seis veces más rápido. Dropout: apagar neuronas al azar durante el entrenamiento, para que las densas no memoricen. Y entrenaron en dos GPU, graphics processing unit, las placas de video, durante entre cinco y seis días. Desde entonces, se entrena en placas de video.

> **Krizhevsky, Sutskever & Hinton, 2012 — ImageNet Classification with Deep Convolutional Neural Networks** · [fuente](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html)
>
> **15,3 % vs. 26,2 %** · Error top-5 en el test de ILSVRC-2012 (promediando 7 redes) contra 26,2 % del segundo. Una sola red: 18,2 %.

El resultado que cambió el campo. En la competencia ILSVRC, ImageNet Large Scale Visual Recognition Challenge, de dos mil doce, AlexNet se equivocó en el quince coma tres por ciento con la métrica top cinco, que cuenta acierto si la clase correcta está entre las cinco más probables. El segundo, con técnicas clásicas, en el veintiséis coma dos. Ojo: el quince coma tres es el promedio de siete redes; una sola daba dieciocho coma dos.

## Resumiendo

- Receta CNN: (conv → ReLU → pool) × n, y densas al final
- LeNet-5: 60.000 parámetros gracias a los pesos compartidos
- CNN chica: 74,7 % con 17 veces menos parámetros que el MLP
- Píxeles mezclados: la CNN cae a 50,1 %; el MLP ni se entera
- Confusión: 268 de 1.000 perros los dice gato
- Próximo: qué aprendió la CNN por dentro